In [2]:
import requests
import json

In [ ]:
# --- STEP 1: Define what we're asking for ---
# Kamloops, BC — one of the most fire-prone areas in the province.
latitude = 50.67
longitude = -120.33

# --- STEP 2: Build the API URL ---
# Open-Meteo's /v1/forecast endpoint provides weather variables, NOT FWI components.
# The FWI (fire_weather_index, ffmc, dmc, dc, isi, bui) are NOT available here —
# you'll need to compute those yourself from the raw weather inputs.
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": ",".join([
        "temperature_2m",          # Air temp at 2m (°C) — FWI input
        "relative_humidity_2m",    # RH at 2m (%) — FWI input
        "wind_speed_10m",          # Wind at 10m (km/h) — FWI input
        "precipitation",           # Rain in mm — FWI input
        # These four variables are the inputs to the Canadian FWI System.
        # You'll calculate FFMC, DMC, DC, ISI, BUI, and FWI from them.
    ]),
    "timezone": "America/Vancouver",  # Get timestamps in BC local time
}

# --- STEP 3: Make the request ---
response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")

if response.status_code != 200:
    print(f"Error: {response.text}")
    exit()

data = response.json()

# --- STEP 4: Check for API-level errors ---
# Open-Meteo returns {"error": true, "reason": "..."} for bad variable names.
# This is what tripped us up before — we asked for variables it doesn't have.
if data.get("error"):
    print(f"API error: {data.get('reason')}")
    exit()

# --- STEP 5: Understand the response structure ---
print("\n--- TOP-LEVEL KEYS ---")
print(list(data.keys()))

print("\n--- METADATA ---")
print(f"  Requested:  ({latitude}, {longitude})")
print(f"  Snapped to: ({data['latitude']}, {data['longitude']})")
print(f"  Elevation:  {data.get('elevation', 'N/A')} m")
print(f"  Timezone:   {data.get('timezone', 'N/A')}")

print("\n--- HOURLY VARIABLES RETURNED ---")
hourly = data["hourly"]
print(f"  Variables: {[k for k in hourly.keys() if k != 'time']}")
print(f"  Time steps: {len(hourly['time'])}")

# --- STEP 6: Look at actual values (first 24 hours) ---
print("\n--- FIRST 24 HOURS OF DATA ---")
print(f"{'Time':<22} {'Temp°C':>7} {'RH%':>5} {'Wind km/h':>10} {'Precip mm':>10}")
print("-" * 60)

for i in range(min(24, len(hourly["time"]))):
    temp = hourly['temperature_2m'][i]
    rh   = hourly['relative_humidity_2m'][i]
    wind = hourly['wind_speed_10m'][i]
    prec = hourly['precipitation'][i]
    print(
        f"{hourly['time'][i]:<22}"
        f"{temp if temp is not None else 'N/A':>7}"
        f"{rh if rh is not None else 'N/A':>5}"
        f"{wind if wind is not None else 'N/A':>10}"
        f"{prec if prec is not None else 'N/A':>10}"
    )

# --- STEP 7: Quick stats ---
temps = [v for v in hourly["temperature_2m"] if v is not None]
rhs   = [v for v in hourly["relative_humidity_2m"] if v is not None]
winds = [v for v in hourly["wind_speed_10m"] if v is not None]
precs = [v for v in hourly["precipitation"] if v is not None]

print("\n--- VALUE RANGES ---")
print(f"  Temp:   {min(temps):.1f} to {max(temps):.1f} °C")
print(f"  RH:     {min(rhs):.0f} to {max(rhs):.0f} %")
print(f"  Wind:   {min(winds):.1f} to {max(winds):.1f} km/h")
print(f"  Precip: {min(precs):.1f} to {max(precs):.1f} mm")
print(f"\n  Total forecast hours: {len(hourly['time'])}")
print(f"  Null count check: temp={sum(1 for v in hourly['temperature_2m'] if v is None)}, "
      f"rh={sum(1 for v in hourly['relative_humidity_2m'] if v is None)}")

print("\n--- NEXT STEP ---")
print("  These 4 weather variables are the inputs to the Canadian FWI System.")
print("  You'll need to compute FFMC, DMC, DC, ISI, BUI, and FWI yourself")
print("  using the Van Wagner (1987) equations or a library like cffdrs/pyfwi.")

Status code: 200

--- TOP-LEVEL KEYS ---
['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']

--- METADATA ---
  Requested:  (50.67, -120.33)
  Snapped to: (50.678204, -120.34752)
  Elevation:  386.0 m
  Timezone:   America/Vancouver

--- HOURLY VARIABLES RETURNED ---
  Variables: ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation']
  Time steps: 168

--- FIRST 24 HOURS OF DATA ---
Time                    Temp°C   RH%  Wind km/h  Precip mm
------------------------------------------------------------
2026-09-15T00:00         14.3   82       7.4       0.0
2026-09-15T01:00         13.3   84       7.7       0.0
2026-09-15T02:00         13.5   83       6.9       0.0
2026-09-15T03:00         13.8   83       6.1       0.0
2026-09-15T04:00         13.1   93       6.1       0.0
2026-09-15T05:00         13.0   93       3.8       0.0
2026-09-15T06:00         12.4   92       4.6   